[Open in Colab](https://colab.research.google.com/github/nz-gravity/sgvb_psd/blob/feature/jax-sgvb/docs/examples/lisa_example_v2.ipynb)

# LISA A/E/T reference fit (parallel Cholesky rows)


For equal-arm second-generation Michelson TDI, use the light-travel time
`tau=L/c`: the common PSD factor is
`16 sin²(2π f tau) sin²(4π f tau)`, not an unsquared sine.
See [Nam et al., Table 2](https://arxiv.org/html/2211.02539v2).
The example reference captures notch locations with a simple smooth envelope
and correlation shape; it is not the complete instrument-noise transfer model.
An explicit positive floor/decorrelation is shown for a valid reference.
Replace `reference_shape` with your full physical `S_ref(f)` when available.

The default downloads/caches a 13.8 MB [truncated noise-4a sample](https://github.com/nz-gravity/test_data/tree/1ed4cde58b9334340208fa17a294e8e7b736b2e9/lisa_noise/noise_4a_truncated/data),
then reads only 4096 rows, split into reference-training and analysis blocks.
Set `DATA_SOURCE="synthetic"` for a no-download model demonstration, or set
`SGVB_LISA_H5` to an existing `X2/Y2/Z2/t` HDF5 file. The full
[noise-4a source](https://zenodo.org/records/15698080) is available separately;
this notebook does not fetch the 505 MB file. A positive broad reference
envelope is fitted on the disjoint training block, with geometry held fixed.
Channel units and the TDI generation must match your chosen reference.


In [ ]:
# Run once in a fresh Colab runtime. This draft API is on the PR branch.
%pip install -q "jax[cpu]==0.6.2" "numpyro==0.19.0" "sgvb_psd[examples] @ git+https://github.com/nz-gravity/sgvb_psd.git@feature/jax-sgvb"


In [ ]:
from pathlib import Path
from time import perf_counter
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sgvb_psd.psd_estimator import PSDEstimator
import jax

plt.rcParams.update({"font.family": "sans-serif", "font.size": 10,
                     "axes.labelsize": 10, "axes.titlesize": 11,
                     "legend.fontsize": 8, "figure.dpi": 110,
                     "text.usetex": False})
print("JAX", jax.__version__, "devices:", jax.devices())
# Deliberately small demonstration budgets, not a convergence certificate.
BUDGET = dict(N_theta=8, nchunks=2, ntrain_map=1000,
              n_elbo_maximisation_steps=300, N_samples=96, seed=7)


In [ ]:
C = 299_792_458.0
ARM_LENGTH = 2.5e9  # metres; update to the geometry/definition of your dataset.
TAU = ARM_LENGTH / C
REFERENCE_FLOOR = 1e-6  # Explicit positive reference floor, not a hidden likelihood floor.

def reference_shape(f):
    x = 2*np.pi*np.asarray(f)*TAU
    cxx = 16*np.sin(x)**2*np.sin(2*x)**2
    # Equal-arm secondary-noise notch shape, with a simple smooth envelope.
    # This is a useful reference, not a complete LISA sensitivity model.
    envelope = 1 + (.003/np.maximum(f, 1e-7))**2
    power = (cxx + REFERENCE_FLOOR)*envelope
    correlation = np.broadcast_to(np.eye(3), (len(f), 3, 3)).copy()
    for i in range(3):
        for j in range(i):
            correlation[:, i, j] = correlation[:, j, i] = -.5*np.cos(x)
    # A small explicit decorrelation of the reference keeps it strictly SPD
    # near a null eigenmode. It is a modelling choice visible here.
    correlation = .999*correlation + .001*np.eye(3)
    return power[:, None, None]*correlation

def simulate_spectrum(spectrum, n, fs, seed):
    rng = np.random.default_rng(seed)
    f = np.fft.rfftfreq(n, 1/fs)
    y = np.zeros((len(f), 3), complex)
    factor = np.linalg.cholesky(fs*spectrum(f[1:-1]))
    noise = (rng.normal(size=(len(f)-2, 3))+1j*rng.normal(size=(len(f)-2, 3)))/np.sqrt(2)
    y[1:-1] = np.einsum("fij,fj->fi", factor, noise)
    return np.fft.irfft(y*np.sqrt(n), n=n, axis=0)


In [ ]:
# Cheap real-data default; explicit offline synthetic option, no silent fallback.
from urllib.request import urlopen
import shutil
DATA_SOURCE = os.environ.get("SGVB_LISA_SOURCE", "sample")
LISA_PATH = Path(os.environ["SGVB_LISA_H5"]) if os.environ.get("SGVB_LISA_H5") else None
if LISA_PATH is None and DATA_SOURCE == "sample":
    LISA_PATH = Path("data/lisa_noise4a_truncated.h5")
    if not LISA_PATH.exists():
        LISA_PATH.parent.mkdir(exist_ok=True)
        url = "https://raw.githubusercontent.com/nz-gravity/test_data/1ed4cde58b9334340208fa17a294e8e7b736b2e9/lisa_noise/noise_4a_truncated/data/tdi.h5"
        temporary = LISA_PATH.with_suffix(".part")
        with urlopen(url, timeout=60) as response, temporary.open("wb") as destination:
            shutil.copyfileobj(response, destination)
        temporary.replace(LISA_PATH)
elif LISA_PATH is None and DATA_SOURCE != "synthetic":
    raise ValueError("DATA_SOURCE must be sample or synthetic")
N_TRAIN = N_FIT = 2048
if LISA_PATH is None:
    fs = .25
    simulation_truth = lambda f: 1e-40*reference_shape(f)
    all_data = simulate_spectrum(simulation_truth, N_TRAIN+N_FIT, fs, 12)
    print("Synthetic LISA-like data; not an instrument-data validation")
else:
    import h5py
    with h5py.File(LISA_PATH, "r") as handle:
        t = np.asarray(handle["t"][:3])
        fs = 1/float(np.median(np.diff(t)))
        all_data = np.column_stack([handle[k][:N_TRAIN+N_FIT] for k in ("X2", "Y2", "Z2")])
    if len(all_data) < N_TRAIN+N_FIT:
        raise ValueError("Need at least 4096 samples")
    simulation_truth = None
    print("Real XYZ2 slice:", LISA_PATH)
training, data = all_data[:N_TRAIN], all_data[N_TRAIN:N_TRAIN+N_FIT]
assert np.all(np.isfinite(all_data))
# Fit three smooth log-envelope coefficients using training only. The
# geometry fixes the dips; exclude near-null training bins where leakage
# dominates instead of trying to fit their depths with a polynomial.
fw, pw = signal.welch(training, fs=fs, axis=0, window="boxcar", nperseg=512,
                       noverlap=0, detrend=False, return_onesided=False)
x = 2*np.pi*fw*TAU
transfer = 16*np.sin(x)**2*np.sin(2*x)**2
keep = (fw > .002) & (fw < min(.1, fs/2)) & (transfer > .1)
shape = reference_shape(fw[keep]).diagonal(axis1=-2, axis2=-1).real
z = np.log(fw[keep]/.02)
design = np.column_stack([np.ones_like(z), z, z*z])
y = np.log(np.median(pw[keep]/shape, axis=1))
assert len(y) > 3 and np.all(np.isfinite(y))
envelope_coefficients = np.linalg.lstsq(design, y, rcond=None)[0]
def reference_xyz(f):
    z = np.log(np.asarray(f)/.02)
    log_envelope = np.column_stack([np.ones_like(z), z, z*z]) @ envelope_coefficients
    return np.exp(log_envelope)[:, None, None]*reference_shape(f)
print("Reference log-envelope coefficients (training only):", envelope_coefficients)


In [ ]:
Q = np.array([[-1, 0, 1], [1, -2, 1], [1, 1, 1]], float)/np.sqrt(np.array([2, 6, 3]))[:, None]
data = data @ Q.T
reference = lambda f: Q @ reference_xyz(f) @ Q.T
truth = (lambda f: Q @ simulation_truth(f) @ Q.T) if simulation_truth is not None else None
labels = ["A", "E", "T"]


In [ ]:
start = perf_counter()
estimator = PSDEstimator(data, fs=fs, reference_psd=reference,
                         parallel_rows=True, fmin_for_analysis=.002, fmax_for_analysis=min(.1, fs/2), **BUDGET)
_ = estimator.run(lr=.003)  # Explicit rate: no costly hyperparameter search.
runtime = perf_counter() - start
assert np.all(np.isfinite(estimator.pointwise_ci))
assert np.all(np.isfinite(estimator.inference_runner.kdl_losses))
print(f"{len(data)} samples × {data.shape[1]} channels; fit {runtime:.2f} s")
if estimator.parallel_rows:
    print("Independent vmapped Cholesky-row dimensions:", estimator.inference_runner.row_dimensions)


In [ ]:
def show_fit(estimator, data, labels, truth=None, reference=None):
    """Two-sided densities and magnitude-squared coherence, no hidden rescaling."""
    from matplotlib.ticker import NullFormatter
    p = data.shape[1]
    fig, axes = plt.subplots(1, p, figsize=(4*p, 3), squeeze=False)
    for i, ax in enumerate(axes[0]):
        fw, pw = signal.welch(data[:, i], fs=estimator.fs, window="boxcar",
                              nperseg=estimator.nt_per_chunk, noverlap=0,
                              detrend=False, return_onesided=False)
        keep = (fw >= estimator.freq[0]) & (fw <= estimator.freq[-1])
        ax.loglog(fw[keep], pw[keep], color="0.65", lw=.7, label="Blocked periodogram")
        q = estimator.pointwise_ci[:, :, i, i].real
        ax.fill_between(estimator.freq, q[0], q[2], alpha=.25)
        ax.loglog(estimator.freq, q[1], label="SGVB median")
        if truth is not None:
            ax.loglog(estimator.freq, truth(estimator.freq)[:, i, i].real, "k--", lw=1, label="Known simulation truth")
        if reference is not None:
            ax.loglog(estimator.freq, reference(estimator.freq)[:, i, i].real, ":", color="C2", lw=1, label="Fixed reference")
        ax.set(title=labels[i], xlabel="Frequency [Hz]", ylabel="Two-sided PSD [input units²/Hz]")
        ticks = np.geomspace(estimator.freq[0], estimator.freq[-1], 4)
        ax.set_xticks(ticks, labels=[f"{f:.2g}" for f in ticks])
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.tick_params(axis="both", labelsize=9)
        ax.legend(fontsize=7)
    fig.tight_layout(); plt.show()
    if p > 1:
        fig, ax = plt.subplots(figsize=(6, 3))
        s = estimator.psd_all
        for i in range(p):
            for j in range(i):
                coh = abs(s[:, :, i, j])**2/(s[:, :, i, i].real*s[:, :, j, j].real)
                q = np.quantile(coh, [.05, .5, .95], axis=0)
                line, = ax.plot(estimator.freq, q[1], label=f"{labels[i]}–{labels[j]}")
                ax.fill_between(estimator.freq, q[0], q[2], alpha=.15, color=line.get_color())
                if truth is not None:
                    known = truth(estimator.freq)
                    known_coh = abs(known[:, i, j])**2/(known[:, i, i].real*known[:, j, j].real)
                    ax.plot(estimator.freq, known_coh, "--", color=line.get_color(), label="Simulation truth")
        ax.set(xlabel="Frequency [Hz]", ylabel="Magnitude-squared coherence", ylim=(0, 1))
        ax.legend(); fig.tight_layout(); plt.show()


In [ ]:
show_fit(estimator, data, labels, truth=truth, reference=reference)


## Interpretation and budget

The shaded bands are pointwise 90% variational intervals. Cheap settings
show the workflow; they do not establish convergence or calibrated coverage.
Increase the data duration, basis size and optimization budgets only for an
explicit analysis/validation plan. Compilation is included in the first-fit
time. JAX batches rows with `vmap`; actual CPU/GPU scheduling is device dependent.
Each row still includes regression on preceding channels, so cross-spectra
are retained. This is modified-Cholesky SGVB, not the removed eigenbasis mode or the
collapsed conditional-theta method in LogPSplinePSD.
